In [4]:
from pylib.setup import *
setup_notebook()

!pip install comext_wrapper --upgrade --quiet

from comext_wrapper import ComextApi
api = ComextApi()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Comext API — lookup notebook

Use this notebook to explore the Eurostat Comext dataset (DS-045409: EU trade by HS2/4/6 and CN8) before building queries.

Three tools:
- `api.info()` — dataset overview and dimension summary
- `api.codes(dimension, search=...)` — browse valid codes
- `api.get_data(...)` — fetch data as a DataFrame

---
## 1. Dataset overview

In [5]:
api.info()

Dataset : DS-045409
Name    : EU trade since 1988 by HS2-4-6 and CN8
Updated : 2026-03-20T11:00:00+0100

 dimension  # codes first code                              first label last code    last label
      freq        9          A                                   Annual         N      Minutely
  reporter      313         AD                                  Andorra        ZW      Zimbabwe
   partner      313         AD                                  Andorra        ZW      Zimbabwe
   product    40409         00 Total for countries whose data are confi  ZZZZZZZZ &lt;No Label>
      flow        3          1                                   IMPORT         3     RE-EXPORT
indicators       44  APQNTBASE Actual production quantity rounding base VALUE_NAC     VALUE_NAC

Key order: freq.reporter.partner.product.flow.indicators


,dimension,# codes,first code,first label,last code,last label
0,freq,9,A,Annual,N,Minutely
1,reporter,313,AD,Andorra,ZW,Zimbabwe
2,partner,313,AD,Andorra,ZW,Zimbabwe
3,product,40409,00,Total for countries whose data are confi,ZZZZZZZZ,&lt;No Label>
4,flow,3,1,IMPORT,3,RE-EXPORT
5,indicators,44,APQNTBASE,Actual production quantity rounding base,VALUE_NAC,VALUE_NAC


In [6]:
api.codes("product")

,id,label
0,00,Total for countries whose data are confidentia...
1,0090,Total for countries whose data are confidentia...
2,009000,Total for countries whose data are confidentia...
3,00900000,Total for countries whose data are confidentia...
4,009000XX,"Trade of subheading 009000, not elsewhere spe..."
...,...,...
40404,99YYY980,"Trade under the declaration limit, SITC 98"
40405,99YYY990,"Trade under the declaration limit, SITC 99"
40406,TO,Total
40407,TOTAL,Total


---
## 2. Browse dimension codes

Valid dimensions: `freq`, `reporter`, `partner`, `product`, `flow`, `indicators`

In [7]:
# Flow and frequency options
print("--- flow ---")
display(api.codes("flow"))

print("--- freq ---")
display(api.codes("freq"))

--- flow ---


,id,label
0,1,IMPORT
1,2,EXPORT
2,3,RE-EXPORT


--- freq ---


,id,label
0,A,Annual
1,S,"Half-yearly, semester"
2,Q,Quarterly
3,M,Monthly
4,W,Weekly
5,D,Daily
6,H,Hourly
7,B,Daily – businessweek
8,N,Minutely


In [8]:
# Partner aggregates — useful for extra/intra-EU splits
api.codes("partner", aggregates_only=True)

,id,label
0,EA19,"Euro area - 19 countries (AT, BE, CY, DE, EE, ..."
1,EA19_EXTRA,Extra-euro area - 19 countries (= 'WORLD' - 'E...
2,EA19_INTRA,"Intra-euro area - 19 countries (AT, BE, CY, DE..."
3,EA20,"Euro area - 20 countries (AT, BE, CY, DE, EE, ..."
4,EA20_EXTRA,Extra-euro area - 20 countries (= 'WORLD' - 'E...
5,EA20_INTRA,"Intra-euro area - 20 countries (AT, BE, CY, DE..."
6,EA21,"Euro area - 21 countries (AT, BE, BG, CY, DE, ..."
7,EA21_EXTRA,Extra-euro area - 21 countries (= 'WORLD' - 'E...
8,EA21_INTRA,"Intra-euro area - 20 countries (AT, BE, BG, CY..."
9,EA_EXTRA,Extra-euro area (= 'WORLD' - 'EA_INTRA')


In [9]:
# Find reporter codes by country name
api.codes("reporter", search="denmark")

,id,label
0,DK,Denmark


In [10]:
# Search product codes by keyword
api.codes("product", search="heat pump")

,id,label
0,841581,Air conditioning machines incorporating a refr...
1,84158100,Air conditioning machines incorporating a refr...
2,84158110,Air conditioning machines incorporating a refr...
3,84158190,Air conditioning machines incorporating a refr...
4,8418,"Refrigerators, freezers and other refrigeratin..."
5,841861,Heat pumps (excl. air conditioning machines of...
6,84186100,Heat pumps (excl. air conditioning machines of...
7,84186910,Refrigerating or freezing equipment and absorp...
8,84186920,Absorption heat pumps
9,84186991,Absorption heat pumps (excl. those for civil a...


In [11]:
# Available value indicators
api.codes("indicators", search="value")

,id,label
0,EXPVAL,"Exports value, euro"
1,IMPVAL,"Imports value, euro"
2,OWNPRODVAL,"Production on own account value, euro"
3,OWNVALBASE,Production on own account value rounding base
4,OWNVALFLAG,Production on own account value flag
5,PRODVAL,"Sold production value, euro"
6,PVALBASE,Sold production value rounding base
7,PVALFLAG,Sold production value flag
8,SCPRODVAL,"Sub-contracted production value, euro"
9,SCVALBASE,Sub-contracted production value rounding base


---
## 3. Fetch data

Use `+` to combine multiple values per dimension.

In [12]:
df = api.get_data(
    reporter     = "DE+DK+FR",
    partner      = "EXT_EU27_2020",   # extra-EU trade
    product      = "84186100",         # heat pumps
    flow         = "1+2",              # imports + exports
    indicators   = "VALUE_IN_EUROS",
    start_period = "2022-01",
    end_period   = "2023-12",
)

df

,freq,reporter,partner,product,flow,indicators,period,value
0,M,DE,EXT_EU27_2020,84186100,1,VALUE_IN_EUROS,2022-01,16915136
1,M,DE,EXT_EU27_2020,84186100,1,VALUE_IN_EUROS,2022-02,16134501
2,M,DE,EXT_EU27_2020,84186100,1,VALUE_IN_EUROS,2022-03,12134974
3,M,DE,EXT_EU27_2020,84186100,1,VALUE_IN_EUROS,2022-04,19023140
4,M,DE,EXT_EU27_2020,84186100,1,VALUE_IN_EUROS,2022-05,19088578
...,...,...,...,...,...,...,...,...
139,M,FR,EXT_EU27_2020,84186100,2,VALUE_IN_EUROS,2023-08,5527840
140,M,FR,EXT_EU27_2020,84186100,2,VALUE_IN_EUROS,2023-09,4491108
141,M,FR,EXT_EU27_2020,84186100,2,VALUE_IN_EUROS,2023-10,6043874
142,M,FR,EXT_EU27_2020,84186100,2,VALUE_IN_EUROS,2023-11,4167049


In [13]:
# Quick pivot: period × reporter, one flow at a time
imports = df[df["flow"] == "1"].pivot_table(
    index="period", columns="reporter", values="value"
)
imports

reporter,DE,DK,FR
period,,,
2022-01,16915136.0,913777.0,31417741.0
2022-02,16134501.0,1746213.0,31166391.0
2022-03,12134974.0,954933.0,24072521.0
2022-04,19023140.0,1775734.0,28566609.0
2022-05,19088578.0,1945158.0,28659968.0
2022-06,18208151.0,1031833.0,19685958.0
2022-07,16162701.0,664162.0,16568435.0
2022-08,19038268.0,1395457.0,20465753.0
2022-09,21105943.0,2249706.0,14518764.0
